## 0. Prérequis : Créer une DB et la table dans postgres

CREATE TABLE reviews (
    id BIGINT PRIMARY KEY,
    listing_id BIGINT,
    date DATE,
    reviewer_id BIGINT,
    reviewer_name TEXT,
    comments TEXT
);


# 1. Création de la DB à partir du fichier reviews.csv

In [7]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import BigInteger, Text, Date

user = "noam"
password = "noam"
host = "localhost"
port = "5432"
database = "projet_big_data"

# Connexion via SQLAlchemy
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")

df = pd.read_csv("../data/source/reviews.csv", parse_dates=["date"])

df.drop_duplicates(subset=["id"], inplace=True)

df.to_sql("reviews", engine, index=False, if_exists="replace", dtype={
    "id": BigInteger(),
    "listing_id": BigInteger(),
    "reviewer_id": BigInteger(),
    "reviewer_name": Text(),
    "comments": Text(),
    "date": Date()
})



849

## 2. Postgres to Bronze


In [29]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("extract reviews from postgres") \
    .config("spark.jars", "/Users/noam/Downloads/postgresql-42.7.4.jar") \
    .enableHiveSupport() \
    .getOrCreate()

df_reviews = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/projet_big_data") \
    .option("dbtable", "reviews") \
    .option("user", "noam") \
    .option("password", "noam") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_reviews.show(5)


+----------+--------+----------+-----------+-------------+--------------------+
|listing_id|      id|      date|reviewer_id|reviewer_name|            comments|
+----------+--------+----------+-----------+-------------+--------------------+
|   7202016|38917982|2015-07-19|   28943674|       Bianca|Cute and cozy pla...|
|   7202016|39087409|2015-07-20|   32440555|        Frank|Kelly has a great...|
|   7202016|39820030|2015-07-26|   37722850|          Ian|Very spacious apa...|
|   7202016|40813543|2015-08-02|   33671805|       George|Close to Seattle ...|
|   7202016|41986501|2015-08-10|   34959538|         Ming|Kelly was a great...|
+----------+--------+----------+-----------+-------------+--------------------+
only showing top 5 rows



### Ajout de la colonne processing date et création de colonne pour le partitionnement

In [31]:
from pyspark.sql.functions import current_date, year, month, dayofmonth

df_reviews = df_reviews.withColumn("processing_date", current_date())
df_reviews = df_reviews \
    .withColumn("year", year("processing_date")) \
    .withColumn("month", month("processing_date")) \
    .withColumn("day", dayofmonth("processing_date"))


### Ecriture en parquet avec partitionnement

In [33]:
df_reviews.write \
    .partitionBy("year", "month", "day") \
    .mode("overwrite") \
    .parquet("../data/bronze/reviews/")


In [27]:
spark.stop()

## 3. Listing

In [35]:
import os
import pandas as pd
 
# Chemin vers le fichier reviews.csv
input_file = "../data/source/listings.csv"
output_dir = "../data/source2/"
batch_size = 100  # Nombre de lignes par lot

# Créer le répertoire "bronze" s'il n'existe pas
os.makedirs(output_dir, exist_ok=True)
 
# Charger le fichier CSV
df = pd.read_csv(input_file)
 
# Diviser le fichier en lots
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i + batch_size]
    batch_file = os.path.join(output_dir, f"reviews_batch_{i // batch_size + 1}.csv")
    batch.to_csv(batch_file, index=False)
 
print(f"Fichier divisé en lots de {batch_size} lignes dans le répertoire {output_dir}.")

Fichier divisé en lots de 100 lignes dans le répertoire ../data/source2/.


In [37]:
from pyspark.sql import SparkSession
import os
 
# Initialisation de la session Spark
#spark = SparkSession.builder \
#    .appName("Copy Source to Bronze") \
#    .getOrCreate()
 
# Chemins des répertoires
source_dir = "../data/source2"
bronze_dir = "../data/bronze/listings"
 
# Vérifier si le répertoire Bronze existe, sinon le créer
os.makedirs(bronze_dir, exist_ok=True)
 
# Lecture des fichiers CSV depuis le répertoire source
df = spark.read.format("csv").option("header", True).load(source_dir)
 
# Écriture des fichiers dans le répertoire Bronze au format Parquet
df.write.format("parquet").mode("overwrite").save(bronze_dir)
 
print(f"Les fichiers ont été copiés de {source_dir} vers {bronze_dir} au format Parquet.")
 
#from pyspark.sql import SparkSession
 
# Initialisation de la session Spark avec support Hive
#spark = SparkSession.builder \
#    .appName("Bronze to Silver with Hive") \
#    .config("spark.sql.catalogImplementation", "hive") \
#    .enableHiveSupport() \
#    .getOrCreate()
 
# Chemins des répertoires
bronze_dir = "../data/bronze"
silver_dir = "../data/silver"
 
# Lecture des fichiers Parquet depuis le répertoire Bronze
df_bronze = spark.read.format("parquet").load(bronze_dir)
 
# Transformation des données (si nécessaire)
# Exemple : Supprimer les lignes avec des valeurs nulles
df_silver = df_bronze.dropna()
 
# Écriture des données dans le répertoire Silver au format Parquet
df_silver.write.format("parquet").mode("overwrite").save(silver_dir)
print(f"Les données ont été écrites dans le répertoire Silver : {silver_dir}")
 
# Écriture des données dans une table Hive
df_silver.write.mode("overwrite").saveAsTable("silver_reviews")
print("Les données ont été chargées dans la table Hive 'silver_reviews'.")
 
# Exemple de requête SQL pour vérifier les données
result = spark.sql("SELECT * FROM silver_reviews LIMIT 10")
result.show()


25/05/02 01:47:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/05/02 01:47:19 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 96,54% for 7 writers
25/05/02 01:47:19 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 84,47% for 8 writers
25/05/02 01:47:20 WARN MemoryManager: Total allocation exceeds 95,00% (906 992 014 bytes) of heap memory
Scaling row group sizes to 96,54% for 7 writers


Les fichiers ont été copiés de ../data/source2 vers ../data/bronze/listings au format Parquet.


Py4JJavaError: An error occurred while calling o144.load.
: java.lang.AssertionError: assertion failed: Conflicting directory structures detected. Suspicious paths:
	file:/users/noam/desktop/airbnb_project/data/bronze/listings
	file:/users/noam/desktop/airbnb_project/data/bronze/reviews

If provided paths are partition directories, please set "basePath" in the options of the data source to specify the root directory of the table. If there are multiple root directories, please load them separately and then union them.
	at scala.Predef$.assert(Predef.scala:223)
	at org.apache.spark.sql.execution.datasources.PartitioningUtils$.parsePartitions(PartitioningUtils.scala:178)
	at org.apache.spark.sql.execution.datasources.PartitioningUtils$.parsePartitions(PartitioningUtils.scala:110)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.inferPartitioning(PartitioningAwareFileIndex.scala:201)
	at org.apache.spark.sql.execution.datasources.InMemoryFileIndex.partitionSpec(In